In [9]:
from pathlib import Path
import json
import pandas as pd
import spacy

DATA_PATH = Path('../data/post-processing/cleaned_outputs.json').resolve()

with DATA_PATH.open() as f:
    raw = json.load(f)

df = pd.DataFrame(raw)
required_cols = ['cleaned_model_output', 'model_type', 'prompt_type', 'seed']
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f'Missing required columns: {missing}')

print(f'Loaded {len(df):,} rows from {DATA_PATH}')
print('Columns:', sorted(df.columns.tolist()))

Loaded 1,440 rows from /mimer/NOBACKUP/groups/naiss2025-22-1187/coherence-driven-humans/data/post-processing/cleaned_outputs.json
Columns: ['cleaned_model_output', 'model_output', 'model_type', 'num_story_images', 'prompt_type', 'seed', 'story_id']


In [ ]:
availability = (
    df.groupby(['prompt_type', 'model_type', 'seed'])
      .size()
      .rename('n')
      .reset_index()
      .sort_values(['prompt_type', 'model_type', 'seed'])
)
availability

,prompt_type,model_type,seed,n
0,large,claude45,42,60
1,large,claude45,43,60
2,large,claude45,44,60
3,large,gpt4o,42,60
4,large,gpt4o,43,60
5,large,gpt4o,44,60
6,large,human,42,60
7,large,human,43,60
8,large,human,44,60
9,large,internvl3,42,60


In [11]:
SEP_TOKEN = '[SEP]'

nlp = spacy.blank('en')
if 'sentencizer' not in nlp.pipe_names:
    nlp.add_pipe('sentencizer')

def split_segments(text: str):
    parts = str(text).split(SEP_TOKEN)
    return [p.strip() for p in parts if p and p.strip()]

def split_sentences(segment: str):
    segment = segment.strip()
    if not segment:
        return []
    doc = nlp(segment)
    return [s.text.strip() for s in doc.sents if s.text.strip()]

def count_words(text: str):
    doc = nlp(str(text))
    return sum(1 for token in doc if not token.is_space and not token.is_punct)

def add_text_metrics(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    out['segments'] = out['cleaned_model_output'].map(split_segments)
    out['n_segments'] = out['segments'].map(len)
    out['segment_sent_counts'] = out['segments'].map(lambda segs: [len(split_sentences(s)) for s in segs])
    out['n_sentences'] = out['segment_sent_counts'].map(sum)
    out['n_words'] = out['cleaned_model_output'].str.replace(SEP_TOKEN, ' ', regex=False).map(count_words)
    return out

dfm = add_text_metrics(df)
dfm[['model_type','prompt_type','seed','n_segments','n_sentences','n_words']].head()

,model_type,prompt_type,seed,n_segments,n_sentences,n_words
0,qwen3vl,original,42,6,7,120
1,qwen3vl,original,42,5,7,144
2,qwen3vl,original,42,5,5,96
3,qwen3vl,original,42,5,5,117
4,qwen3vl,original,42,5,12,134


In [12]:
def keep_row(row):
    model = row['model_type']
    prompt = row['prompt_type']
    seed = int(row['seed'])

    if prompt == 'original':
        if model == 'claude45':
            return seed == 44
        return seed == 42

    if prompt == 'large':
        if model == 'human':
            return seed in {42, 43, 44}
        return seed == 42

    return False

subset = dfm[dfm.apply(keep_row, axis=1)].copy()
print(f'Subset rows kept: {len(subset):,}')
subset.groupby(['prompt_type','model_type','seed']).size().rename('n').reset_index().sort_values(['prompt_type','model_type','seed'])

Subset rows kept: 840


,prompt_type,model_type,seed,n
0,large,claude45,42,60
1,large,gpt4o,42,60
2,large,human,42,60
3,large,human,43,60
4,large,human,44,60
5,large,internvl3,42,60
6,large,llama4scout,42,60
7,large,qwen3vl,42,60
8,original,claude45,44,60
9,original,gpt4o,42,60


In [13]:
def summarize(group: pd.DataFrame) -> pd.Series:
    n_docs = len(group)
    total_segments = group['n_segments'].sum()
    total_sentences = group['n_sentences'].sum()
    total_words = group['n_words'].sum()

    return pd.Series({
        'Docs': n_docs,
        'Seg/Doc': group['n_segments'].mean() if n_docs else 0.0,
        'Sent/Seg': (total_sentences / total_segments) if total_segments else 0.0,
        'Sent/Doc': group['n_sentences'].mean() if n_docs else 0.0,
        'Words/Doc': group['n_words'].mean() if n_docs else 0.0,
        'Words/Sent': (total_words / total_sentences) if total_sentences else 0.0,
        'Words/Seg': (total_words / total_segments) if total_segments else 0.0,
    })

def pretty(tbl: pd.DataFrame):
    out = tbl.copy()
    out['Docs'] = out['Docs'].astype(int)
    for c in ['Seg/Doc','Sent/Seg','Sent/Doc','Words/Doc','Words/Sent','Words/Seg']:
        out[c] = out[c].round(2)
    return out

def with_group(frame: pd.DataFrame):
    out = frame.copy()
    out['group'] = out['model_type'].map(lambda m: 'Human' if m == 'human' else 'LLMs')
    return out

In [14]:
# Table A: Human vs LLMs per prompt type
table_prompt_group = (
    with_group(subset)
    .groupby(['prompt_type', 'group'], sort=True)
    .apply(summarize)
    .reset_index()
    .rename(columns={'prompt_type': 'Prompt', 'group': 'System'})
)
table_prompt_group = pretty(table_prompt_group)
table_prompt_group

,Prompt,System,Docs,Seg/Doc,Sent/Seg,Sent/Doc,Words/Doc,Words/Sent,Words/Seg
0,large,Human,180,5.70,2.12,12.07,164.91,13.66,28.93
1,large,LLMs,300,5.93,1.74,10.34,203.02,19.63,34.24
2,original,Human,60,5.80,1.29,7.48,76.48,10.22,13.19
3,original,LLMs,300,5.71,1.98,11.32,176.77,15.62,30.98


In [15]:
# Table B: Per-prompt per-system detail (Human + each model separately)
table_prompt_model = (
    subset.groupby(['prompt_type', 'model_type'], sort=True)
          .apply(summarize)
          .reset_index()
          .rename(columns={'prompt_type': 'Prompt', 'model_type': 'System'})
)
table_prompt_model = pretty(table_prompt_model)
table_prompt_model

,Prompt,System,Docs,Seg/Doc,Sent/Seg,Sent/Doc,Words/Doc,Words/Sent,Words/Seg
0,large,claude45,60,5.75,2.14,12.28,277.47,22.59,48.26
1,large,gpt4o,60,5.73,1.70,9.77,148.92,15.25,25.97
2,large,human,180,5.70,2.12,12.07,164.91,13.66,28.93
3,large,internvl3,60,6.12,2.26,13.80,224.02,16.23,36.62
4,large,llama4scout,60,6.32,1.44,9.10,196.33,21.58,31.08
5,large,qwen3vl,60,5.73,1.18,6.77,168.38,24.88,29.37
6,original,claude45,60,5.67,2.09,11.82,209.68,17.74,37.00
7,original,gpt4o,60,5.60,2.21,12.37,163.65,13.23,29.22
8,original,human,60,5.80,1.29,7.48,76.48,10.22,13.19
9,original,internvl3,60,5.95,2.47,14.68,220.77,15.04,37.10


In [ ]:
table_original = table_prompt_model[table_prompt_model['Prompt'] == 'original'].drop(columns=['Prompt']).reset_index(drop=True)
table_large = table_prompt_model[table_prompt_model['Prompt'] == 'large'].drop(columns=['Prompt']).reset_index(drop=True)

print('Original prompt table')
display(table_original)
print('Large prompt table')
display(table_large)

Original prompt table


,System,Docs,Seg/Doc,Sent/Seg,Sent/Doc,Words/Doc,Words/Sent,Words/Seg
0,claude45,60,5.67,2.09,11.82,209.68,17.74,37.00
1,gpt4o,60,5.60,2.21,12.37,163.65,13.23,29.22
2,human,60,5.80,1.29,7.48,76.48,10.22,13.19
3,internvl3,60,5.95,2.47,14.68,220.77,15.04,37.10
4,llama4scout,60,5.65,1.99,11.25,181.30,16.12,32.09
5,qwen3vl,60,5.67,1.14,6.48,108.45,16.73,19.14


Large prompt table


,System,Docs,Seg/Doc,Sent/Seg,Sent/Doc,Words/Doc,Words/Sent,Words/Seg
0,claude45,60,5.75,2.14,12.28,277.47,22.59,48.26
1,gpt4o,60,5.73,1.70,9.77,148.92,15.25,25.97
2,human,180,5.70,2.12,12.07,164.91,13.66,28.93
3,internvl3,60,6.12,2.26,13.80,224.02,16.23,36.62
4,llama4scout,60,6.32,1.44,9.10,196.33,21.58,31.08
5,qwen3vl,60,5.73,1.18,6.77,168.38,24.88,29.37
